# 🛒 Ejercicios: eBay
# Subastas y ventas - precio, pujas y conversión

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de eBay.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué categoría vende
### mejor? (tasa de
### conversión real)

Conversión = % de artículos vendidos.
Lo que no se vende, no factura.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/ebay_subastas.csv'
)

r = (
  df
  .groupby('categoria')['vendido']
  .mean()
  .mul(100)
  .round(1)
  .sort_values(ascending=False)
)

print('Tasa de venta por categoría:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto revaloriza
### el artículo cada
### puja extra?

Agrupamos por tramos de pujas y
vemos el precio final medio.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/ebay_subastas.csv'
)

v = df[
  (df['vendido'])
  & (df['tipo_venta'] == 'Subasta')
].copy()

v['tramo'] = pd.cut(
  v['pujas'],
  bins=[-1, 2, 5, 10, 100],
  labels=['0-2', '3-5',
          '6-10', '10+'],
)

r = (
  v
  .groupby('tramo',
    observed=True)
  ['precio_final_usd']
  .mean()
  .round(2)
)

print('Precio final por tramo pujas:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### Subasta vs Compra ya:
### ¿qué formato factura
### más por artículo?

Comparamos ingreso medio (solo
vendidos) y tasa de venta.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/ebay_subastas.csv'
)

r = (
  df
  .groupby('tipo_venta')
  .agg(
    conversion=('vendido',
                'mean'),
    precio=('precio_final_usd',
            lambda s:
            s[s > 0].mean()),
  )
)

r['conversion'] = (
  r['conversion'] * 100
).round(1)
r['precio'] = r['precio'].round(2)

print('Por formato de venta:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿El envío gratis
### ayuda a vender
### de verdad? (lift)

Comparamos conversión con y sin
envío gratis. Lift = cuánto sube.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/ebay_subastas.csv'
)

r = (
  df
  .groupby('envio')['vendido']
  .mean()
)

lift = (
  r['Gratis'] / r['Pago'] - 1
) * 100

print('Conversión por envío:')
print((r * 100).round(1))
print(f'\nLift envío gratis: '
      f'+{lift:.1f}%')

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué categorías mueven
### el grueso de la
### facturación? (Pareto)

Suma del precio final vendido por
categoría y su % acumulado.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/ebay_subastas.csv'
)

s = (
  df
  .groupby('categoria')
  ['precio_final_usd']
  .sum()
  .sort_values(ascending=False)
)

acum = (
  s.cumsum() / s.sum() * 100
).round(1)

tabla = pd.DataFrame({
  'factur_usd': s.round(0),
  '%_acum': acum,
})

print('Pareto de facturación:')
print(tabla)